# OpenVLA en MuJoCo — comportamiento mínimo (Colab)

Este notebook carga `openvla/openvla-7b-finetuned-libero-10` en **4-bit** (para caber en la **T4** del free tier), lo conecta con la simulación del **Panda** y muestra un video corto del brazo movido por las acciones del modelo.

> **Es una prueba de *plumbing***: verifica que el modelo carga, produce acciones y el brazo se mueve. **No** es una tarea de LIBERO resuelta: para eso hay que alinear la escena/cámara/acción al setup exacto de LIBERO (objetos, cámara agentview, convención de pinza y frame). Aquí la cámara y la escena son genéricas.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`.

## 1. Render headless (EGL) + dependencias
`MUJOCO_GL=egl` debe fijarse **antes** de importar mujoco.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"   # render sin pantalla (Colab)

# Simulación
!pip -q install mujoco imageio imageio-ffmpeg
# Stack de OpenVLA. Versiones fijadas por compatibilidad con el codigo remoto.
# (sin flash-attn: la T4 no lo soporta)
!pip -q install "transformers==4.40.1" "timm==0.9.10" tokenizers accelerate bitsandbytes
print("dependencias instaladas")

## 2. Dónde se guardan los pesos

El checkpoint pesa **~16 GB**, así que **no cabe en Drive** (con 1 GB libre). Se descargan al **disco local de Colab** (~78 GB, efímero): se re-descargan cada sesión (unos minutos), pero caben sin problema. **No** montamos Drive.

In [ ]:
# Los pesos van al disco local efimero de Colab (se re-descargan por sesion).
# Cache de HuggingFace en disco local (NO en Drive):
os.environ["HF_HOME"] = "/content/hf_cache"
print("HF_HOME =", os.environ["HF_HOME"])
print("Espacio libre en disco local:")
!df -h /content | tail -1

## 3. Subir el proyecto
Sube un **.zip** de tu carpeta `MuJoCo/` que incluya `robots/`, `simulation.py`, `openvla_controller.py` y `MJCF/panda/` (con su `assets/`).

_Alternativa: si lo tienes en GitHub, reemplaza esta celda por_ `!git clone <tu-repo>`.

In [ ]:
import sys, glob, zipfile
from google.colab import files
print("Sube el .zip del proyecto...")
up = files.upload()
zname = next(n for n in up if n.lower().endswith(".zip"))
zipfile.ZipFile(zname).extractall("/content/proj")
hits = glob.glob("/content/proj/**/simulation.py", recursive=True)
assert hits, "No encontre simulation.py dentro del zip"
ROOT = os.path.dirname(hits[0])
sys.path.insert(0, ROOT); os.chdir(ROOT)
print("Proyecto en:", ROOT)

## 4. Sanity check de la simulación (sin el modelo aún)
Construye el Panda y muestra la observación que verá OpenVLA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from robots import RobotFactory, Frame
from simulation import Simulation

sim = Simulation(RobotFactory.create("panda"))
camera = sim.default_camera(azimuth=135, elevation=-20)
obs = sim.observation(camera, 224, 224)
plt.imshow(obs); plt.axis("off"); plt.title(f"Observacion {obs.shape}"); plt.show()

## 5. Cargar OpenVLA (4-bit)
Descarga ~16 GB (cada sesion, sin cache en Drive) y cuantiza a 4-bit al vuelo. Tarda unos minutos.

In [ ]:
from openvla_controller import OpenVLAController

vla = OpenVLAController(
    model_id="openvla/openvla-7b-finetuned-libero-10",
    load_in_4bit=True,
    device="cuda",
    frame=Frame.WORLD,          # a calibrar
    gripper_open_is_high=True,  # a calibrar
)
print("modelo cargado | unnorm_key =", vla.unnorm_key)

## 6. Lazo cerrado: observar → predecir → actuar → avanzar
El modelo emite una acción por paso; la aplicamos con `cartesian_step` y avanzamos la física con un `frame_skip` pequeño. Guardamos la vista del modelo como video.

In [ ]:
import imageio

instruction = "pick up the object and place it"   # texto libre (smoke test)
sim.reset()
frames = []
for t in range(60):
    obs = sim.observation(camera, 224, 224)
    action = vla.predict(obs, instruction)                 # -> CartesianAction
    sim.robot.controller.cartesian_step(action, frame=vla.frame)
    sim.step(5)                                            # frame_skip
    frames.append(obs)

imageio.mimsave("/content/openvla_demo.mp4", frames, fps=10)
print("listo:", len(frames), "frames")

In [ ]:
from IPython.display import Video
Video("/content/openvla_demo.mp4", embed=True, width=360)

## Notas / próximos pasos
- **Esto es plumbing, no una tarea resuelta.** Para que el modelo se comporte bien hay que **alinear la escena a LIBERO**: cámara *agentview* en la pose correcta, objetos de la tarea, y verificar la convención de **frame** (`Frame.WORLD` vs `Frame.TOOL`) y de **pinza** (`gripper_open_is_high`). Se calibran cambiando esos parámetros al crear `OpenVLAController` y observando si el brazo va "al derecho".
- **`unnorm_key`**: si la carga se queja, pásalo explícito. Puedes inspeccionar `vla.model.norm_stats.keys()`.
- **Rendimiento**: en T4 sin flash-attn, ~1–2 acciones/s. Normal.
- **Añadir un objeto**: para una tarea real, agrega un cubo con free-joint + mesa a la escena del Panda y aleatoriza su pose en `reset`.